# 6. Validação cruzada manual

Amostra estratificada de itens pra conferir manualmente contra uma fonte
externa curada (ex.: paldb.cc). Não é validação automatizável - o objetivo
aqui é gerar uma visualização fácil de auditar e um lugar pra registrar o
resultado da conferência.


In [1]:
import json
from pathlib import Path

import pandas as pd

# Notebook lives in notebooks/etl/, so the repo root is two levels up.
REPO_ROOT = Path("../..").resolve()

ROOT = REPO_ROOT / "data/Pal/Content"
PAL = ROOT / "Pal"
ITEM_DT = PAL / "DataTable/Item/DT_ItemDataTable_Common.json"
RECIPE_DT = PAL / "DataTable/Item/DT_ItemRecipeDataTable_Common.json"
BUILDOBJECT_DT = PAL / "DataTable/MapObject/Building/DT_BuildObjectDataTable_Common.json"
BENCH_RECIPES = REPO_ROOT / "src/data/bench_recipes.json"
NAMES_DT_EN = ROOT / "L10N/en/Pal/DataTable/Text/DT_ItemNameText_Common.json"
NAMES_DT_PT_BR = ROOT / "L10N/pt-BR/Pal/DataTable/Text/DT_ItemNameText_Common.json"


def load_rows(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)[0]["Rows"]


In [2]:
items_raw = load_rows(ITEM_DT)
recipes_raw = load_rows(RECIPE_DT)
buildings_raw = load_rows(BUILDOBJECT_DT)

items_df = pd.DataFrame.from_dict(items_raw, orient="index")
recipes_df = pd.DataFrame.from_dict(recipes_raw, orient="index")
buildings_df = pd.DataFrame.from_dict(buildings_raw, orient="index")

print(f"items: {items_df.shape}, recipes: {recipes_df.shape}, buildings: {buildings_df.shape}")


items: (2466, 53), recipes: (1414, 20), buildings: (498, 32)


In [3]:
# Padroniza a string sentinela "None" (usada pela UE pra ausência de valor)
# para o None real do Python, e monta um resolvedor de id case-insensitive -
# ver 02_limpeza.ipynb pro raciocínio completo por trás disso.
items_clean = items_df.replace("None", None)
recipes_clean = recipes_df.replace("None", None)
buildings_clean = buildings_df.replace("None", None)

items_by_lower = {item_id.lower(): item_id for item_id in items_clean.index}


def resolve_item_id(raw_id):
    # pandas' .replace("None", None) upcasts these cells to NaN (a float),
    # not Python None - "is None" or plain truthiness checks silently miss
    # it and .lower() blows up on a float. pd.isna() catches both.
    if pd.isna(raw_id):
        return None
    if raw_id in items_clean.index:
        return raw_id
    return items_by_lower.get(raw_id.lower())


## Helpers: nome de exibição e árvore de ingredientes de 1 nível


In [4]:
names_en = load_rows(NAMES_DT_EN)


def display_name(item_id):
    row = names_en.get(f"ITEM_NAME_{item_id}")
    return row["TextData"]["LocalizedString"] if row else item_id


def recipe_ingredients(item_id):
    # Retorna (product_count, {ingredient: material_count cru}) - sem normalizar,
    # igual a scripts/crafting_graph.py: a receita produz product_count por craft.
    if item_id not in recipes_clean.index:
        return 1, {}
    recipe = recipes_clean.loc[item_id]
    product_count = recipe["Product_Count"] or 1
    out = {}
    for i in range(1, 6):
        material_id = recipe[f"Material{i}_Id"]
        material_count = recipe[f"Material{i}_Count"] or 0
        if pd.isna(material_id) or material_count == 0:
            continue
        resolved = resolve_item_id(material_id)
        if resolved:
            out[resolved] = material_count
    return product_count, out


## Amostra estratificada

Até 2 itens craftáveis por `TypeA`, escolhidos de forma determinística
(sem RNG) pra reprodutibilidade entre execuções.


In [5]:
craftable = items_clean[items_clean.index.isin(recipes_clean.index)]
sample_ids = craftable.groupby("TypeA").head(2).index.tolist()
print(f"{len(sample_ids)} item(ns) selecionado(s) para conferência manual")


24 item(ns) selecionado(s) para conferência manual


## Árvore de ingredientes de cada item da amostra (conferir contra a fonte externa)


In [6]:
for item_id in sample_ids:
    product_count, ingredients = recipe_ingredients(item_id)
    print(f"\n{item_id} ({display_name(item_id)}) - produz {product_count} por craft")
    for ing_id, qty in ingredients.items():
        print(f"  {qty:g}x {ing_id} ({display_name(ing_id)})")



Money (Gold Coin) - produz 20000 por craft
  30x CopperIngot (Ingot)

Arrow (Arrow) - produz 10 por craft
  2x Wood (Wood)
  2x Stone (Stone)

Arrow_Poison (Poison Arrow) - produz 10 por craft
  2x Wood (Wood)
  2x Stone (Stone)
  1x Venom (Venom Gland)

AssaultRifle_Default1 (Assault Rifle) - produz 1 por craft
  40x IronIngot (Refined Ingot)
  10x Polymer (Polymer)
  10x CarbonFiber (Carbon Fiber)

Axe_Tier_00 (Stone Axe) - produz 1 por craft
  5x Stone (Stone)
  5x Wood (Wood)

Baked_Berries (Baked Berries) - produz 1 por craft
  1x Berries (Red Berries)

Charcoal (Charcoal) - produz 1 por craft
  2x Wood (Wood)

ClothArmor (Cloth Outfit) - produz 1 por craft
  2x Cloth (Cloth)
  7x Fiber (Fiber)

CopperArmor (Metal Armor) - produz 1 por craft
  30x CopperIngot (Ingot)
  10x Leather (Leather)
  5x Cloth (Cloth)

Herbs (Low Grade Medical Supplies) - produz 1 por craft
  5x Berries (Red Berries)
  2x Horn (Horn)

Medicines (Medical Supplies) - produz 1 por craft
  3x CopperIngot (Ing

## Tabela de conferência (preencher à mão)

Copiar os itens da amostra acima, comparar cada receita com paldb.cc (ou
outra fonte curada) e preencher esta tabela.

| item_id | nome | receita esperada (fonte externa) | receita no dado | bate? | nota |
|---|---|---|---|---|---|
| | | | | | |
